# Series 2.3 — RAG Chunking

**Why AI Fails? — Engineering Lab**

---

> Most RAG apps ask: *What is the right chunk size?*  
> This lab answers with **benchmarks**, not framework defaults.

**Scenario:** Same documentation corpus, same questions, same model — only **how documents are split** changes.

**Core lesson:** Better RAG is not retrieving **more** context. It is retrieving the **right evidence** at the **lowest useful cost**.


## 1. The Problem

| Bad RAG assumption | Engineering approach |
|--------------------|----------------------|
| "Use 512 tokens because the tutorial said so" | Benchmark small / medium / large / semantic on **your** corpus |
| Retrieve more chunks = better answers | Hit Score + prompt tokens show the tradeoff |
| One chunk size fits all questions | Chunk according to **how users ask** |
| Ignore cost until the bill arrives | Measure tokens, latency, and estimated cost per strategy |

### Why this matters in production

- Wrong chunk size → **missed evidence** (too small) or **bloated prompts** (too large)
- Every extra token in retrieved chunks is billed **on every question**
- Retrieval quality and cost move together — you must measure both

**Expected dry-run insight:**

```
SMALL   → lower prompt tokens, may miss cross-section context
MEDIUM  → often best balance for documentation-style corpora
LARGE   → higher hit scores sometimes, higher prompt cost always
SEMANTIC→ structure-aware sections when docs have clear headings
```


## 2. What is RAG Chunking?

**RAG chunking** is how you **split source documents** into retrievable pieces before embedding or keyword search.

It is not about changing the LLM, the retriever algorithm, or the user question. It is about **defining the unit of evidence** the model reads.

### Definition

```
RAG chunking = documents → chunks → retrieve top-K → prompt → LLM
               (chunk size is the first cost/quality lever)
```

### Chunk size tradeoffs

| Smaller chunks | Larger chunks |
|----------------|---------------|
| Lower prompt cost per retrieval | More context per chunk |
| Precise keyword hits | May include irrelevant padding |
| May split related facts across chunks | Higher token bill |
| Good for fact lookup | Good for narrative / comparison questions |

### What RAG chunking is NOT

| Technique | Difference |
|-----------|------------|
| **Context pruning** (Series 2.1) | Pruning filters **one request's** evidence; chunking defines **document structure** |
| **Prompt caching** (Series 2.2) | Caching reuses stable instructions; chunks are **changing retrieved content** |
| **Embedding model choice** | Chunking happens **before** vectors are created |
| **Top-K tuning** | Top-K chooses **how many** chunks; chunking defines **what each chunk contains** |

### This lab's implementation

```
docs/ corpus
    → chunker.py (small / medium / large / semantic)
    → retriever.py (keyword + Hit Score)
    → prompt_builder.py → Gemini
```

> **Enterprise principle:** Benchmark chunk strategies on real questions — don't copy framework defaults.


## 3. Repository Layout

```
why-ai-fails/
├── docs/                          ← Article corpus for RAG benchmark
│   ├── series_2_hidden_economics.txt
│   ├── series_2_1_context_pruning.txt
│   ├── series_2_2_prompt_caching.txt
│   └── series_2_3_rag_chunking.txt
├── common/                        ← Shared Gemini client, token math
└── series-2.3/
    ├── app.py                     ← CLI + benchmark runner
    ├── chunker.py                 ← Fixed-size + semantic chunking
    ├── retriever.py               ← Keyword scoring + Hit Score
    ├── questions.py               ← Benchmark Q&A + expected terms
    ├── prompt_builder.py          ← RAG prompt template
    ├── benchmark.py               ← Side-by-side comparison
    ├── README.md
    └── Series_2.3_RAG_Chunking.ipynb   ← This notebook
```


## 4. Python Files in This Lab

Every `.py` file under `series-2.3/`:

| File | What it does |
|------|--------------|
| **`app.py`** | CLI entry point. Loads `docs/` corpus, runs all chunking strategies and benchmark questions, calls Gemini (or `--dry-run`), prints comparison. |
| **`chunker.py`** | Document splitting — fixed-size strategies (`small`/`medium`/`large` with overlap) and `semantic` (split on `#` headings). Exposes `STRATEGIES` config. |
| **`retriever.py`** | Keyword retriever (no vector DB). `score_chunk()` ranks by term overlap, `retrieve_top_k()` selects chunks, `compute_hit_score()` measures retrieval quality. |
| **`questions.py`** | Benchmark questions (`q1`–`q3`) with `expected_terms` for Hit Score. |
| **`prompt_builder.py`** | `build_rag_prompt()` — assembles user question + labeled retrieved chunks into the final Gemini prompt. |
| **`benchmark.py`** | Side-by-side strategy printer — Hit Score, prompt tokens, latency, cost. |


## 5. The Chunking Pipeline (`chunker.py`)

```
docs/*.txt
    │
    ▼  Strategy: small (200 tokens, 0 overlap)
    ▼  Strategy: medium (500 tokens, 50 overlap)
    ▼  Strategy: large (1000 tokens, 100 overlap)
    ▼  Strategy: semantic (split on # headings)
    │
    ▼  retriever.py → top-K chunks + Hit Score
    ▼  prompt_builder.py → Gemini (or --dry-run)
```

| Strategy | Chunk size | Overlap | Expected behavior |
|----------|------------|---------|-------------------|
| `small` | 200 tokens | 0 | Lower cost, may miss cross-section context |
| `medium` | 500 tokens | 50 | Balanced retrieval and cost |
| `large` | 1000 tokens | 100 | More context per chunk, higher prompt cost |
| `semantic` | by `#` headings | — | Structure-aware sections |

**Hit Score** = matched expected terms / total expected terms (retrieval quality proxy).


## 6. Three Layers of RAG Engineering

### Layer 1 — Same corpus, same questions (controlled experiment)

Every strategy runs on **identical** documents and benchmark questions. Only chunking changes — so differences in Hit Score and tokens are attributable.

---

### Layer 2 — Retrieve the right evidence (not all evidence)

Keyword retriever scores chunks, selects top-K, and builds a prompt with **only** those chunks.

```
Question → tokenize → score all chunks → top-K → prompt
```

The anti-pattern: dump entire documents into every prompt "just in case."

---

### Layer 3 — Measure (Hit Score + tokens + cost)

| Metric | What it tells you |
|--------|-------------------|
| **Hit Score** | Did retrieved chunks contain expected terms? |
| **Prompt tokens** | Cost driver — scales with chunk size × top-K |
| **Latency / Cost** | From `common/token_usage.py` |

| Mode | Flag | API key? |
|------|------|----------|
| **Dry-run** | `--dry-run` | No — Hit Score + token estimates, **$0** |
| **Live** | (none) | Yes — real Gemini answers per strategy |


## 7. Execution Flow

```
Parse CLI args (--strategy, --question-id, --top-k)
    │
    └─ Load docs/ corpus
            │
            For each chunking strategy:
                chunk documents
                retrieve top-K for each benchmark question
                compute Hit Score + token estimate
                → Gemini (unless --dry-run)
            │
            └─ print_benchmark() — side-by-side comparison
```


## 8. How to Run

From the **repo root**:

```bash
pip install -r requirements.txt
cp .env.example .env   # optional — only for live Gemini calls
```

| Command | What it does | API key? |
|---------|--------------|----------|
| `python series-2.3/app.py --dry-run` | All strategies, all questions | No |
| `python series-2.3/app.py --strategy medium --dry-run` | Single strategy | No |
| `python series-2.3/app.py --question-id q1 --dry-run` | Single question | No |
| `python series-2.3/app.py` | Live Gemini benchmark | Yes |


In [ ]:
# Live demo cell — run the dry-run benchmark ($0, no API key needed)
# Execute this cell during your presentation

import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "demo.py").exists() and (ROOT.parent / "demo.py").exists():
    ROOT = ROOT.parent

result = subprocess.run(
    [sys.executable, str(ROOT / "series-2.3/app.py"), "--dry-run"],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
print(f"\nExit code: {result.returncode}")


## 9. Key Code Snippets

### Chunk strategies (`chunker.py`)

```python
STRATEGIES = {
    "small":  {"chunk_size": 200,  "overlap": 0,   "mode": "fixed"},
    "medium": {"chunk_size": 500,  "overlap": 50,  "mode": "fixed"},
    "large":  {"chunk_size": 1000, "overlap": 100, "mode": "fixed"},
    "semantic": {"mode": "semantic"},  # split on # headings
}
```

### Hit Score (`retriever.py`)

```python
def compute_hit_score(retrieved_text: str, expected_terms: list[str]) -> float:
    matched = sum(1 for t in expected_terms if t.lower() in retrieved_text.lower())
    return matched / len(expected_terms)
```


## 10. Where Series 2.3 Fits

| Lab | Topic | Builds on prior labs by… |
|-----|-------|--------------------------|
| 2.1 | Context Pruning | Shrinks evidence before the prompt |
| 2.2 | Prompt Caching | Reuses stable system instructions |
| **2.3** | **RAG Chunking** | Defines **how knowledge is split** for retrieval |
| 2.4 | Conversation Summarization | Compresses chat history |
| 2.5 | Long-Term Memory | Stores durable user facts |
| 2.6 | Memory Retrieval | Finds the right memory at scale |
| 2.7 | Model Routing | Selects the right model per request |

---

## Takeaway

> **Chunk according to how users ask questions — not framework defaults.**

**Next lab:** [Series 2.4 — Conversation Summarization](../series-2.4/) — long chat sessions need memory management, not full history replay.
